In [14]:
# Tiler layers use jupyter-xarray-tiler (TiTiler + Jupyter /proxy/...) via jupytergis_lab.
from jupytergis import GISDocument
from typing import Iterable, List
from pyproj import Transformer
import pystac_client
import stackstac

def extent_3857_to_bbox_4326(extent: Iterable[float]) -> List[float]:
    """
    Convert a Web Mercator extent to a STAC-ready lon/lat bbox.
    Input extent format: [minx, miny, maxx, maxy] in EPSG:3857
    Output bbox format:  [min_lon, min_lat, max_lon, max_lat] in EPSG:4326
    """
    minx, miny, maxx, maxy = extent
    transformer = Transformer.from_crs(
        "EPSG:3857",
        "EPSG:4326",
        always_xy=True,  # keep x=lon, y=lat ordering
    )
    min_lon, min_lat = transformer.transform(minx, miny)
    max_lon, max_lat = transformer.transform(maxx, maxy)
    return [min_lon, min_lat, max_lon, max_lat]

doc = GISDocument("footprints.jGIS")

In [15]:
m = doc.extent
# Full map extent in lon/lat (for reference only).
bbox_doc = (
    extent_3857_to_bbox_4326([m.west, m.south, m.east, m.north]) if m else None
)
# STAC search and stackstac.stack must use the *same* bounds; otherwise the
# stack is huge and mostly empty while items only cover a small area.
bbox_search = [ -18.248776440117897, 27.004514659408795, -16.033094271404384, 34.01156206721229 ]
# [
#     -4.85,   # west  (min lon)
#     47.56,    # south (min lat)
#     -3.99,   # east  (max lon)
#     47.93,     # north (max lat)
# ] 
# [-2.98, 47.50, -2.58, 47.66]

bbox = bbox_search
bbox_doc, bbox

([-3.6097577297544787,
  46.251993084994815,
  2.1730648605939207,
  48.423465019798236],
 [-18.248776440117897,
  27.004514659408795,
  -16.033094271404384,
  34.01156206721229])

In [16]:
# URL = "https://stac.dataspace.copernicus.eu/v1"
URL = "https://earth-search.aws.element84.com/v1"
catalog = pystac_client.Client.open(URL)

In [17]:
items = catalog.search(
    bbox=bbox,
    collections=["sentinel-2-l2a"],
    datetime="2020-03-01/2020-03-02",
).item_collection()
len(items)

42

In [18]:
da = stackstac.stack(items, epsg=4326, bounds_latlon=bbox)
da

<xarray.DataArray 'stackstac-657b0038b5366bf1e3c92acd9dd1e5eb' (time: 42,
                                                                band: 32,
                                                                y: 94820,
                                                                x: 107244)> Size: 109TB
dask.array<fetch_raster_window, shape=(42, 32, 94820, 107244), dtype=float64, chunksize=(1, 1, 1024, 1024), chunktype=numpy.ndarray>
Coordinates: (12/52)
  * time                                     (time) datetime64[ns] 336B 2020-...
    id                                       (time) <U24 4kB 'S2B_28SDB_20200...
    earthsearch:boa_offset_applied           (time) bool 42B False True ... True
    s2:not_vegetated_percentage              (time) object 336B 0.001468 ... 0
    s2:high_proba_clouds_percentage          (time) float64 336B 15.36 ... 0....
    s2:datatake_id                           (time) <U34 6kB 'GS2B_20200301T1...
    ...                                       ...
    s2:reflectance_conversion_factor         float64 8B 1.02
    s2:product_type                          <U7 28B 'S2MSI2A'
    s2:saturated_defective_pixel_percentage  int64 8B 0
    instruments                              <U3 12B 'msi'
    platform                                 <U11 44B 'sentinel-2b'
    epsg                                     int64 8B 4326
Attributes:
    spec:           RasterSpec(epsg=4326, bounds=(-18.248781978205, 27.004476...
    crs:            epsg:4326
    transform:      | 0.00, 0.00,-18.25|\n| 0.00,-0.00, 34.01|\n| 0.00, 0.00,...
    resolution_xy:  (2.0660339438867353e-05, 7.389942854740193e-05)

In [ ]:
state = {"lazy": None, "computed": None}

def ndvi_process(sliced):
    red = sliced.sel(band='red')
    nir = sliced.sel(band='nir')
    out = (nir - red) / (nir + red)
    out = out.where((nir + red) != 0)
    out.name = 'ndvi'
    return out

def ndwi_process(sliced):
    green = sliced.sel(band='green')
    nir = sliced.sel(band='nir')
    ndwi = (green - nir) / (green + nir)
    ndwi = ndwi.where((green + nir) != 0)
    ndwi.name = 'ndwi'
    if "time" in sliced.dims:
        print("slicing time")
        return ndwi.mean(dim="time", skipna=True, keep_attrs=True)
    return ndwi

sub_id = doc.bind_extent_slice_callback(
    da,
    on_slice=lambda out: state.update({"lazy": out}),  # lazy dask-backed DataArray
    data_crs='EPSG:4326',
    x_name='x',
    y_name='y',
    process=ndwi_process,
    compute=True,  # <-- keep lazy
    strict_process=False,  # keep state updates even if process throws
    viewport_width=640,
    viewport_height=360,
    resample_method='linear',
    auto_viewport_from_options=False,
    add_tiler_layer_on_compute=True,
    replace_previous_tiler_layer=True,
    tiler_layer_kwargs={
        'name': 'NDWI viewport',
        'colormap_name': 'viridis',
        'rescale': (-1.0, 1.0),
        'opacity': 0.9,
    },
)

In [19]:
doc

In [11]:
state["lazy"]

In [ ]:
import dask.diagnostics

lazy = state["lazy"]
assert lazy is not None, "Pan/zoom the map (or set extent) so the callback runs."
with dask.diagnostics.ProgressBar():
    state["computed"] = lazy.compute()


In [ ]:
import math
import numpy as np

c = state["computed"]
if "time" in c.dims:
    finite_fracs = []
    for t in range(c.sizes["time"]):
        v = c.isel(time=t).values
        finite_fracs.append(float(np.isfinite(v).mean()))
    best_t = int(np.argmax(finite_fracs))
    img = c.isel(time=best_t)
else:
    best_t = None
    img = c

vals = img.values
finite = np.isfinite(vals)
if finite.any():
    vmin = float(np.nanpercentile(vals, 2))
    vmax = float(np.nanpercentile(vals, 98))
else:
    vmin, vmax = -0.5, 0.5

if (not math.isfinite(vmin)) or (not math.isfinite(vmax)) or (vmin >= vmax):
    vmin, vmax = -0.5, 0.5

(best_t, img.dims, img.shape, (vmin, vmax))


In [ ]:
await doc.add_tiler_layer(
    state["lazy"],
    name='ndwi-manual-compute',
    colormap_name='plasma',
    rescale=(-1, 1),
    reproject='max',
)

In [ ]:
doc.unbind_extent_slice_callback(sub_id)
